# 🔷 Vectors and Transformations

**One transformation, expanded to reveal the hidden machinery of linear algebra in algorithms.**

---

## The Hook
What does a matrix *do*?

## 1. Concept Intuition

A **vector** is an arrow — it has direction and magnitude. In 2D, it's just two numbers: how far right, how far up.

A **matrix** is a *machine*. You feed it a vector, and it spits out a different vector. The matrix defines the transformation — rotation, scaling, shearing, reflection.

$$\begin{bmatrix} a & b \\ c & d \end{bmatrix} \begin{bmatrix} x \\ y \end{bmatrix} = \begin{bmatrix} ax + by \\ cx + dy \end{bmatrix}$$

This is the key insight: **a matrix IS a function**. Not a table. Not storage. A function.

### Why this matters for DSA

- An **adjacency matrix** encodes a graph — matrix[i][j] = 1 means "edge from i to j"
- **Matrix multiplication** counts paths — $A^k[i][j]$ = number of paths of length $k$ from $i$ to $j$
- **Transformation chaining** = matrix multiplication — applying transform B then A is just $A \cdot B$

## 2. Visual Representation

Let's see what a matrix *does* to space.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
%matplotlib inline

def plot_transformation(M, title='', ax=None):
    """Visualize what a 2x2 matrix does to the unit square and basis vectors."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    
    # Unit square corners
    square = np.array([[0,0], [1,0], [1,1], [0,1], [0,0]]).T  # 2×5
    transformed = M @ square
    
    # Draw original (faded)
    ax.fill(square[0], square[1], alpha=0.1, color='gray')
    ax.plot(square[0], square[1], 'k--', alpha=0.3, linewidth=1)
    
    # Draw transformed
    ax.fill(transformed[0], transformed[1], alpha=0.25, color='#6366f1')
    ax.plot(transformed[0], transformed[1], color='#6366f1', linewidth=2)
    
    # Basis vectors: original
    ax.annotate('', xy=(1, 0), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5, alpha=0.4))
    ax.annotate('', xy=(0, 1), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5, alpha=0.4))
    
    # Basis vectors: transformed
    e1 = M @ np.array([1, 0])
    e2 = M @ np.array([0, 1])
    ax.annotate('', xy=e1, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='#f43f5e', lw=2.5))
    ax.annotate('', xy=e2, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='#10b981', lw=2.5))
    
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)
    ax.set_title(title, fontsize=12, fontweight='bold')
    return ax

# Show four fundamental transformations
fig, axes = plt.subplots(2, 2, figsize=(11, 11))

transforms = [
    (np.array([[2, 0], [0, 0.5]]), 'Scale: stretch x, compress y'),
    (np.array([[np.cos(np.pi/4), -np.sin(np.pi/4)], 
               [np.sin(np.pi/4),  np.cos(np.pi/4)]]), 'Rotate: 45° counter-clockwise'),
    (np.array([[1, 1], [0, 1]]), 'Shear: skew along x'),
    (np.array([[1, 0], [0, -1]]), 'Reflect: flip over x-axis'),
]

for ax, (M, title) in zip(axes.flat, transforms):
    plot_transformation(M, title, ax)

plt.suptitle('One Unit Square → Four Transformations', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Interactive Experiment

### Build your own transformation

In [ ]:
from ipywidgets import interact, FloatSlider

@interact(
    a=FloatSlider(value=1, min=-3, max=3, step=0.1, description='a'),
    b=FloatSlider(value=0, min=-3, max=3, step=0.1, description='b'),
    c=FloatSlider(value=0, min=-3, max=3, step=0.1, description='c'),
    d=FloatSlider(value=1, min=-3, max=3, step=0.1, description='d')
)
def transform_explorer(a, b, c, d):
    M = np.array([[a, b], [c, d]])
    det = np.linalg.det(M)
    
    # Transform unit square and circle
    theta = np.linspace(0, 2*np.pi, 100)
    circle = np.array([np.cos(theta), np.sin(theta)])
    transformed = M @ circle
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Unit square transformation
    plot_transformation(M, f'Matrix [[{a},{b}],[{c},{d}]]', axes[0])
    
    # Circle transformation
    axes[1].plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.3, linewidth=1)
    axes[1].plot(transformed[0], transformed[1], color='#f43f5e', linewidth=2)
    axes[1].set_xlim(-4, 4); axes[1].set_ylim(-4, 4)
    axes[1].set_aspect('equal')
    axes[1].grid(True, alpha=0.3)
    axes[1].axhline(y=0, color='k', linewidth=0.5)
    axes[1].axvline(x=0, color='k', linewidth=0.5)
    axes[1].set_title(f'Circle → Ellipse (det = {det:.2f})', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # What kind of transformation is this?
    if abs(det) < 0.01:
        print('⚠ Determinant ≈ 0: Transformation collapses space (singular matrix)')
    elif det < 0:
        print(f'↺ Determinant < 0: Orientation is flipped, area scaled by {abs(det):.2f}')
    else:
        print(f'→ Determinant = {det:.2f}: Area scaled by {det:.2f}, orientation preserved')

### Rotation animation

## The Explanation (ADEPT Method)

Let's break it down...

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(6, 6))
theta_pts = np.linspace(0, 2*np.pi, 100)

# A house shape to transform
house = np.array([[0, 1, 1, 0.5, 0, 0],
                  [0, 0, 0.8, 1.2, 0.8, 0]])

line_orig, = ax.plot(house[0], house[1], 'k--', alpha=0.3, linewidth=1)
line_trans, = ax.plot([], [], color='#6366f1', linewidth=2.5)

ax.set_xlim(-2, 2); ax.set_ylim(-2, 2)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

def animate(frame):
    angle = frame * np.pi / 30
    R = np.array([[np.cos(angle), -np.sin(angle)],
                  [np.sin(angle),  np.cos(angle)]])
    rotated = R @ house
    line_trans.set_data(rotated[0], rotated[1])
    ax.set_title(f'Rotation: {np.degrees(angle):.0f}°', fontsize=13, fontweight='bold')
    return line_trans,

ani = FuncAnimation(fig, animate, frames=60, interval=80, blit=False)
plt.close(fig)
HTML(ani.to_jshtml())

## 3. Mathematical Formulation

### Linear transformation

A matrix $A \in \mathbb{R}^{m \times n}$ defines a linear transformation $T: \mathbb{R}^n \to \mathbb{R}^m$:

$$T(\mathbf{v}) = A\mathbf{v}$$

**Key properties of linearity:**
- $T(\mathbf{u} + \mathbf{v}) = T(\mathbf{u}) + T(\mathbf{v})$ — transformation distributes over addition
- $T(c\mathbf{v}) = cT(\mathbf{v})$ — scaling commutes with transformation

### Rotation matrix

$$R(\theta) = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}$$

### Composition = multiplication

Applying $B$ then $A$:  $T_{total}(\mathbf{v}) = A(B\mathbf{v}) = (AB)\mathbf{v}$

### Determinant = area scaling factor

$$\det(A) = ad - bc \quad \text{for } A = \begin{bmatrix} a & b \\ c & d \end{bmatrix}$$

If $|\det(A)| = 2$, the transformation doubles areas. If $\det(A) < 0$, it flips orientation.

## 4. Code Implementation

### Matrix as function: transforming a cloud of points

In [ ]:
# Generate points on a circle
theta = np.linspace(0, 2*np.pi, 100)
circle = np.array([np.cos(theta), np.sin(theta)])  # 2×100 matrix of points

# Define a transformation
M = np.array([[2, 1],
              [0.5, 1.5]])

# Transform ALL points at once (matrix multiplication)
transformed = M @ circle

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(circle[0], circle[1], color='#6366f1', linewidth=2)
axes[0].set_title('Original circle', fontsize=12, fontweight='bold')
axes[0].set_aspect('equal')
axes[0].set_xlim(-4, 4); axes[0].set_ylim(-4, 4)
axes[0].grid(True, alpha=0.3)

axes[1].plot(transformed[0], transformed[1], color='#f43f5e', linewidth=2)
axes[1].set_title(f'After M (det = {np.linalg.det(M):.2f})', fontsize=12, fontweight='bold')
axes[1].set_aspect('equal')
axes[1].set_xlim(-4, 4); axes[1].set_ylim(-4, 4)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Matrix transforms a circle into an ellipse', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Matrix:\n{M}')
print(f'Determinant: {np.linalg.det(M):.4f} (area scaling factor)')
print(f'Eigenvalues: {np.linalg.eigvals(M)}')

### Graph as matrix: adjacency matrix encodes structure

In [ ]:
# A simple directed graph as an adjacency matrix
# Nodes: 0, 1, 2, 3
# Edges: 0→1, 0→2, 1→2, 2→3, 3→0
A = np.array([[0, 1, 1, 0],
              [0, 0, 1, 0],
              [0, 0, 0, 1],
              [1, 0, 0, 0]])

# A² counts paths of length 2
A2 = A @ A
# A³ counts paths of length 3
A3 = A @ A @ A

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (mat, title) in zip(axes, [(A, 'A: direct edges'),
                                     (A2, 'A²: paths of length 2'),
                                     (A3, 'A³: paths of length 3')]):
    im = ax.imshow(mat, cmap='YlOrRd', vmin=0, vmax=max(mat.max(), 1))
    ax.set_title(title, fontsize=11, fontweight='bold')
    for i in range(4):
        for j in range(4):
            ax.text(j, i, str(mat[i, j]), ha='center', va='center', fontsize=14,
                   color='white' if mat[i,j] > mat.max()/2 else 'black')
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    ax.set_xlabel('To node'); ax.set_ylabel('From node')

plt.suptitle('Matrix Powers = Path Counting', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('A²[0][3] =', A2[0][3], '→ number of paths of length 2 from node 0 to node 3')
print('A³[0][0] =', A3[0][0], '→ number of cycles of length 3 starting at node 0')

## 6. Tool Exploration

Understanding NumPy's linear algebra toolkit.

In [ ]:
M = np.array([[2, 1], [1, 3]])
v = np.array([1, 2])

print('=== Matrix operations ===')
print(f'M @ v (transform):  {M @ v}')
print(f'M @ M (compose):    \n{M @ M}')
print(f'det(M):             {np.linalg.det(M):.4f}')
print(f'M.T (transpose):    \n{M.T}')
print()

print('=== Eigendecomposition ===')
eigenvalues, eigenvectors = np.linalg.eig(M)
print(f'Eigenvalues:  {eigenvalues}')
print(f'Eigenvectors: \n{eigenvectors}')
print()

# Verify: M @ eigenvector = eigenvalue * eigenvector
for i in range(len(eigenvalues)):
    ev = eigenvectors[:, i]
    lhs = M @ ev
    rhs = eigenvalues[i] * ev
    print(f'M @ e{i} = {lhs},  λ{i} * e{i} = {rhs},  match: {np.allclose(lhs, rhs)}')

print()
print('=== Inverse ===')
M_inv = np.linalg.inv(M)
print(f'M⁻¹:\n{M_inv}')
print(f'M @ M⁻¹ = Identity: {np.allclose(M @ M_inv, np.eye(2))}')

## 7. Reflection Questions

1. In the interactive explorer, what matrix values produce a **pure rotation** (no scaling)? What condition must the matrix satisfy?

2. Set the determinant to 0 (try `a=1, b=2, c=0.5, d=1`). What happens to the shape? Why is this matrix called "singular"? What does this mean for a system of equations?

3. If $A[i][j]$ is an adjacency matrix, what does $(A^2)[i][j]$ count? Can you generalize to $A^k$? How could you use this to check if a graph is connected?

4. Why is matrix multiplication not commutative ($AB \neq BA$ in general)? Create two matrices where the order of application produces visibly different results.

5. Eigenvectors are directions that "survive" a transformation (they only get scaled). In the context of a graph's adjacency matrix, what do eigenvectors represent? (Hint: think PageRank.)

---

*You now see structure. Next: embrace randomness. → `Randomness_and_Expectation/`*

## 🔬 Break-It Lab
(Exercises merged from earlier structure)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math101.testing import check_answer, difficulty_banner, score_report
from math101.tracker import record_score

results = []

## 🟢 Warm-Up: Recall & Predict

## 🟡 Challenge: Compute & Verify

## 🔴 Boss Level: Build From Scratch

## 📊 Results

In [ ]:
score_report(results)
record_score('Structure_and_Transformations', 'exercises', sum(results), len(results))

## The Feynman Technique
Explain [this concept] in plain English to someone who has never seen it. Write it in the cell below. Then check: did you use any jargon you can't define from scratch? If yes, go back.

*(Write your explanation here...)*

## Review
- **Q:** 
- **A:** 

- **Q:** 
- **A:** 

## The Takeaway
> **The one thing to carry forward is:** *(Write the insight, not the formula)*